# Prepare the ISLES'24 Fold 4 handoff for Kaggle

This notebook needs **CPU only**. It packages the already-persistent nnU-Net preprocessed data plus Fold 4 checkpoint state from Google Drive into a ~3–4 GiB private handoff archive, then uploads it to Kaggle as a **private Dataset**.

The medical/challenge data must not be published. The Kaggle CLI creates datasets as private by default; this notebook never passes `--public`.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')


## Load the latest repository


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')
if REPO.exists(): shutil.rmtree(REPO)
if REPO_ZIP.exists(): REPO_ZIP.unlink()
subprocess.run([
    'curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o',str(REPO_ZIP)
],check=True)
subprocess.run(['unzip','-q',str(REPO_ZIP),'-d','/content'],check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-q','-e','.'],check=True)


## Build the minimal resume bundle in Drive


In [ ]:
ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
HANDOFF_DIR = ROOT / 'kaggle_handoff'
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python','scripts/make_isles2024_kaggle_resume_bundle.py',
    '--preprocessed-root',str(ROOT/'nnUNet_preprocessed'),
    '--results-root',str(ROOT/'nnUNet_results'),
    '--output-dir',str(HANDOFF_DIR),
    '--fold','4'
],check=True)

ARCHIVE = HANDOFF_DIR/'isles2024_fold4_resume.tar'
print('Bundle:', ARCHIVE)
print('Size GiB:', round(ARCHIVE.stat().st_size/(1024**3),2))


## Authenticate to Kaggle

Download a Kaggle API token from your Kaggle account settings (`kaggle.json`), then run this cell and upload that tiny JSON file when prompted.


In [ ]:
uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise RuntimeError('Please upload kaggle.json')
KAGGLE_DIR = Path.home()/'.kaggle'
KAGGLE_DIR.mkdir(exist_ok=True)
(KAGGLE_DIR/'kaggle.json').write_bytes(uploaded['kaggle.json'])
os.chmod(KAGGLE_DIR/'kaggle.json',0o600)
subprocess.run(['python','-m','pip','install','-q','kaggle'],check=True)
subprocess.run(['kaggle','config','view'],check=True)


## Upload the handoff as a private Kaggle Dataset


In [ ]:
credentials = json.loads((KAGGLE_DIR/'kaggle.json').read_text())
username = credentials['username']
UPLOAD_DIR = Path('/content/kaggle_upload')
if UPLOAD_DIR.exists(): shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir()
shutil.copy2(ARCHIVE,UPLOAD_DIR/ARCHIVE.name)
metadata = {
    'title':'ISLES2024 NCCT Fold4 Resume - Private',
    'id':f'{username}/isles2024-ncct-fold4-resume',
    'licenses':[{'name':'other'}]
}
(UPLOAD_DIR/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2))

check = subprocess.run(['kaggle','datasets','files',metadata['id']],capture_output=True,text=True)
if check.returncode == 0:
    subprocess.run(['kaggle','datasets','version','-p',str(UPLOAD_DIR),'-m','Updated Fold 4 resume state','-r','skip'],check=True)
else:
    subprocess.run(['kaggle','datasets','create','-p',str(UPLOAD_DIR),'-r','skip'],check=True)
print('Private Kaggle Dataset:', metadata['id'])
